# 🤖 AI IT Helpdesk Agent — Version 4
## Human-in-the-Loop + Ticket Creation + Memory

---

Welcome to **Version 4** of our AI IT Helpdesk Agent series!  

In this notebook, we build on top of V3's **Agent + Tool Calling Loop** by adding three powerful new capabilities:

1. **🎫 Ticket Creation** — The agent can create IT support tickets on behalf of the user  
2. **🛑 Human-in-the-Loop (HITL)** — The graph *pauses* to ask for human approval before creating tickets  
3. **🧠 Conversation Memory** — The agent remembers previous messages within a session using `MemorySaver`

---

### 🗺️ Series Roadmap

| Version | Focus | Key Concepts |
|---------|-------|-------------|
| **V1** | Manual Conditional Routing | `StateGraph`, conditional edges, basic routing |
| **V2** | LLM-Powered Classification | LLM decides the route, structured output |
| **V3** | Agent + Tool Calling Loop | `ToolNode`, `tools_condition`, autonomous agent loop |
| **V4** ✅ | **HITL + Tickets + Memory** | **`interrupt()`, `Command(resume=...)`, `MemorySaver`** |

---

## 🔄 What's New in V4 vs V3?

| Feature | V3 | V4 |
|---------|-----|-----|
| Agent + Tool Loop | ✅ | ✅ |
| Tools: password, VPN, general IT | ✅ | ✅ |
| `create_ticket` tool | ❌ | ✅ **NEW** |
| `view_tickets` tool | ❌ | ✅ **NEW** |
| In-memory ticket store | ❌ | ✅ **NEW** |
| Human-in-the-Loop (`interrupt`) | ❌ | ✅ **NEW** |
| Approval / Rejection flow | ❌ | ✅ **NEW** |
| Conversation Memory (`MemorySaver`) | ❌ | ✅ **NEW** |
| Multi-turn conversations | ❌ | ✅ **NEW** |
| `thread_id` config | ❌ | ✅ **NEW** |

---

### 🧩 How HITL Works (Interrupt Flow)

```
User: "Create a ticket for my flickering monitor"
        │
        ▼
   ┌─────────┐
   │  Agent   │ ──▶ Decides to call create_ticket tool
   └─────────┘
        │
        ▼
   ┌─────────┐
   │  Tools   │ ──▶ create_ticket runs... hits interrupt()
   └─────────┘
        │
     ⏸️ PAUSE  ──▶ Graph freezes, state saved to checkpointer
        │
  [Human reviews the ticket details]
        │
  Command(resume="yes")  or  Command(resume="no")
        │
        ▼
   ┌─────────┐
   │  Tools   │ ──▶ create_ticket resumes with the answer
   └─────────┘
        │
        ▼
   ┌─────────┐
   │  Agent   │ ──▶ Formats final response to user
   └─────────┘
```

---

## 📦 Step 1: Install Dependencies

In [ ]:
%pip install -U langgraph langchain-openai python-dotenv

---

## 📥 Step 2: Imports

We bring in everything from V3, plus three **new** imports:

- **`MemorySaver`** — A checkpointer that stores graph state in memory, enabling conversation persistence and HITL  
- **`interrupt`** — A function that *pauses* the graph mid-execution and waits for human input  
- **`Command`** — Used to *resume* a paused graph with a value (e.g., `"yes"` or `"no"`)

In [ ]:
import os
import random
import json
from datetime import datetime

# --- LangGraph Core ---
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# --- NEW in V4 ---
from langgraph.checkpoint.memory import MemorySaver    # Checkpointer for memory & HITL
from langgraph.types import interrupt, Command          # HITL: pause & resume

# --- LangChain ---
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

# --- Environment ---
from dotenv import load_dotenv

print("✅ All imports successful!")

---

## 🔑 Step 3: Load Environment Variables

In [ ]:
load_dotenv()

print("✅ Environment variables loaded!")

---

## 🧠 Step 4: Create the LLM

We initialize ChatOpenAI with `temperature=0` for deterministic, reliable responses.  
The model name is read from the `.env` file, defaulting to `gpt-4.1-mini`.

In [ ]:
model = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
    temperature=0
)

print(f"✅ LLM initialized: {model.model_name}")

---

## 🎫 Step 5: Ticket Storage System (NEW)

This is a **simple in-memory ticket store** — a Python list that holds ticket dictionaries.  

In a production system, this would be replaced with a database (e.g., PostgreSQL, MongoDB, or a ticketing API like Jira/ServiceNow).  
For our purposes, this lightweight store lets us demonstrate the full ticket lifecycle:

- **Create** tickets via the `create_ticket` tool  
- **View** tickets via the `view_tickets` tool  
- **Persist** tickets across conversations within the same notebook session

In [ ]:
# ============================================================
#  In-Memory Ticket Store
# ============================================================

TICKET_STORE = []


def save_ticket(ticket_id: str, summary: str, priority: str) -> dict:
    """Save a new ticket to the in-memory store."""
    ticket = {
        "ticket_id": ticket_id,
        "summary": summary,
        "priority": priority,
        "status": "open",
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    TICKET_STORE.append(ticket)
    return ticket


def get_all_tickets() -> list:
    """Retrieve all tickets from the store."""
    return TICKET_STORE


print("✅ Ticket store initialized (empty)")
print(f"   Current tickets: {len(TICKET_STORE)}")

---

## 🔧 Step 6: Define Tools

We define **5 tools** total — 3 returning from V3, plus 2 brand-new ones:

| Tool | Source | Description |
|------|--------|-------------|
| `password_help` | V3 | Password reset & account lockout guidance |
| `vpn_help` | V3 | VPN connection troubleshooting |
| `general_it_help` | V3 | Catch-all for other IT issues |
| `create_ticket` | **NEW** | Creates a support ticket (with HITL approval!) |
| `view_tickets` | **NEW** | Lists all open tickets |

### 6a. Standard Tools (from V3)

In [ ]:
# ============================================================
#  Tool 1: Password Help
# ============================================================

@tool
def password_help(issue: str) -> str:
    """Help users with password resets, account lockouts, and login issues.
    
    Args:
        issue: Description of the password or login problem.
    """
    return (
        "Password Reset Steps:\n"
        "1. Go to https://passwordreset.company.com\n"
        "2. Enter your employee ID\n"
        "3. Verify your identity via email or SMS\n"
        "4. Create a new password (min 12 chars, 1 uppercase, 1 number, 1 symbol)\n"
        "5. If account is locked, wait 30 minutes or contact IT at ext. 4357\n\n"
        "⏱️ Lockout auto-clears after 30 minutes."
    )


# ============================================================
#  Tool 2: VPN Help
# ============================================================

@tool
def vpn_help(issue: str) -> str:
    """Help users troubleshoot VPN connection problems.
    
    Args:
        issue: Description of the VPN problem.
    """
    return (
        "VPN Troubleshooting Steps:\n"
        "1. Ensure you're connected to the internet first\n"
        "2. Restart the VPN client (GlobalProtect / Cisco AnyConnect)\n"
        "3. Try a different VPN server region\n"
        "4. Disable any firewall or antivirus temporarily\n"
        "5. Clear VPN cache: Settings → VPN → Clear Credentials\n"
        "6. If still failing, restart your computer and try again\n\n"
        "🔗 VPN Portal: https://vpn.company.com"
    )


# ============================================================
#  Tool 3: General IT Help
# ============================================================

@tool
def general_it_help(issue: str) -> str:
    """Handle general IT support questions including hardware, software,
    email, printing, and other technical issues.
    
    Args:
        issue: Description of the IT problem.
    """
    return (
        "General IT Support:\n"
        "1. Restart your device (fixes ~70%% of issues)\n"
        "2. Check for pending software updates\n"
        "3. Clear browser cache if it's a web app issue\n"
        "4. Check https://status.company.com for known outages\n"
        "5. If the issue persists, you can create a support ticket\n\n"
        "📧 IT Support Email: itsupport@company.com\n"
        "📞 IT Hotline: ext. 4357"
    )


print("✅ Standard tools defined: password_help, vpn_help, general_it_help")

### 6b. 🆕 `create_ticket` Tool — With Human-in-the-Loop!

This is the **star of V4**.  

When the agent decides to create a ticket, this tool **pauses the entire graph** using `interrupt()`.  
The human gets to review the ticket details (summary, priority) and either **approve** or **reject** it.

> **How `interrupt()` works:**
> 1. The tool calls `interrupt({...})` with the details to show the human  
> 2. LangGraph **freezes** execution and saves the full state to the checkpointer  
> 3. The developer/user inspects the interrupt and sends `Command(resume="yes")` or `Command(resume="no")`  
> 4. The tool **resumes** exactly where it left off — the return value of `interrupt()` is the resume value

In [ ]:
# ============================================================
#  Tool 4: Create Ticket (NEW — with HITL interrupt)
# ============================================================

@tool
def create_ticket(issue_summary: str, priority: str = "medium") -> str:
    """Create an IT support ticket for unresolved issues.
    Use when user explicitly asks to create a support ticket,
    or when troubleshooting steps did not resolve the issue.

    Args:
        issue_summary: Brief description of the issue.
        priority: Ticket priority level. Options: low, medium, high.
    """

    # ---- HITL: Pause and ask for human approval ----
    approval = interrupt({
        "question": "Do you approve creating this support ticket?",
        "summary": issue_summary,
        "priority": priority
    })

    # ---- Resume: Process the human's decision ----
    if approval == "yes":
        ticket_id = f"IT-{random.randint(1000, 9999)}"
        save_ticket(ticket_id, issue_summary, priority)
        return (
            f"✅ Ticket {ticket_id} created successfully!\n"
            f"Summary: {issue_summary}\n"
            f"Priority: {priority}"
        )
    else:
        return "❌ Ticket creation cancelled by user."


print("✅ create_ticket tool defined (with interrupt)")

### 6c. 🆕 `view_tickets` Tool

A simple read-only tool that lists all tickets in the store.  
The agent uses this when the user asks to see their tickets or check on ticket status.

In [ ]:
# ============================================================
#  Tool 5: View Tickets (NEW)
# ============================================================

@tool
def view_tickets() -> str:
    """View all open IT support tickets.
    Use when the user asks to see their tickets or check ticket status."""

    tickets = get_all_tickets()

    if not tickets:
        return "No tickets found."

    result = "Open Tickets:\n\n"
    for t in tickets:
        result += (
            f"  {t['ticket_id']} | "
            f"{t['priority'].upper()} | "
            f"{t['summary']} | "
            f"{t['created_at']}\n"
        )

    return result


print("✅ view_tickets tool defined")

---

## 🔗 Step 7: Bind Tools to the LLM

We register all 5 tools with the model so it knows what's available to call.

In [ ]:
# Collect all tools into a single list
tools = [
    password_help,
    vpn_help,
    general_it_help,
    create_ticket,
    view_tickets
]

# Bind tools to the model
model_with_tools = model.bind_tools(tools)

print("✅ Tools bound to LLM:")
for t in tools:
    print(f"   🔧 {t.name}")

---

## 💬 Step 8: System Prompt (Updated for V4)

The system prompt now includes instructions about **ticket creation** and **ticket viewing** — two capabilities the agent didn't have in V3.

In [ ]:
# ============================================================
#  System Prompt — V4 (Updated)
# ============================================================

SYSTEM_PROMPT = """You are an AI IT Helpdesk Agent for a large company.

Your capabilities:
- Help users reset passwords and resolve account lockouts
- Troubleshoot VPN connection issues
- Handle general IT support questions (hardware, software, email, printing)
- **Create IT support tickets** when issues cannot be resolved through troubleshooting
- **View open tickets** when users ask about their ticket status

Guidelines:
1. Always be polite, professional, and empathetic.
2. Use the appropriate tool for each type of issue.
3. Provide clear, step-by-step instructions.
4. If troubleshooting doesn't resolve the issue, offer to create a support ticket.
5. When creating tickets, summarize the issue clearly and assign appropriate priority:
   - **high**: System down, security breach, cannot work at all
   - **medium**: Degraded functionality, intermittent issues
   - **low**: Nice-to-have, cosmetic issues, feature requests
6. Always confirm ticket details before creation.
7. You can view existing tickets when asked.
"""

print("✅ System prompt configured")

---

## 🤖 Step 9: Define the Agent Node

Same structure as V3 — the agent node injects the system prompt and invokes the LLM with tools.

In [ ]:
# ============================================================
#  Agent Node
# ============================================================

def agent_node(state: MessagesState):
    """The agent node: injects system prompt, calls LLM with tools."""

    # Prepend system message to the conversation
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]

    # Call the LLM (may produce a tool call or a direct response)
    response = model_with_tools.invoke(messages)

    return {"messages": [response]}


print("✅ Agent node defined")

---

## 🏗️ Step 10: Build the Graph

The graph structure is **identical to V3** — an agent node and a tools node connected in a loop.  
The magic of HITL comes from the `interrupt()` inside the `create_ticket` tool, not from the graph structure itself.

```
        ┌──────────┐
START → │  Agent   │
        └────┬─────┘
             │
        tools_condition
        ╱            ╲
   has tools       no tools
      │               │
      ▼               ▼
 ┌─────────┐        END
 │  Tools  │
 └────┬────┘
      │
      └──────▶ Agent (loop back)
```

In [ ]:
# ============================================================
#  Build the StateGraph
# ============================================================

builder = StateGraph(MessagesState)

# Add nodes
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))

# Add edges
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

print("✅ Graph structure built")

---

## 🧠 Step 11: Compile with MemorySaver (NEW)

This is a **critical difference from V3**.  In V3 we compiled with just `builder.compile()`.  
In V4, we pass a **checkpointer** — `MemorySaver`.

### Why do we need a checkpointer?

Two reasons:

1. **Human-in-the-Loop** — When `interrupt()` pauses the graph, the entire state (messages, tool calls, etc.) must be saved somewhere. The checkpointer stores this state so the graph can be resumed later with `Command(resume=...)`.

2. **Conversation Memory** — By using a `thread_id` in the config, subsequent `.invoke()` calls on the same thread automatically pick up the previous messages. The agent "remembers" what you talked about before.

> **Note:** `MemorySaver` stores everything in Python memory (a dictionary). It's perfect for development and testing. For production, you'd use `SqliteSaver`, `PostgresSaver`, or another persistent backend.

In [ ]:
# ============================================================
#  Compile with Checkpointer (NEW in V4)
# ============================================================

memory = MemorySaver()

helpdesk_graph = builder.compile(checkpointer=memory)

print("✅ Graph compiled with MemorySaver checkpointer")
print("   → HITL (interrupt/resume) is now enabled")
print("   → Conversation memory is now enabled")

---

## 📊 Step 12: Visualize the Graph

In [ ]:
from IPython.display import Image, display

display(
    Image(helpdesk_graph.get_graph().draw_mermaid_png())
)

---

## 🧪 Testing the Agent

We'll run **5 tests** to demonstrate every new capability:

| Test | What it demonstrates |
|------|---------------------|
| Test 1 | Normal VPN query (same as V3, but with `thread_id`) |
| Test 2 | Ticket creation with **approval** (HITL ✅) |
| Test 3 | Ticket creation with **rejection** (HITL ❌) |
| Test 4 | Viewing tickets |
| Test 5 | Multi-turn conversation **memory** |

---

### 🧪 Test 1: Normal VPN Query

This is the same type of query we tested in V3.  
The key difference: we now pass a **`config`** dict with a **`thread_id`**.  
This is required because we have a checkpointer — every invocation must belong to a thread.

In [ ]:
# ============================================================
#  Test 1: Normal VPN Query (non-HITL)
# ============================================================

config = {"configurable": {"thread_id": "test-vpn"}}

result = helpdesk_graph.invoke(
    {"messages": [{"role": "user", "content": "My VPN is not connecting"}]},
    config
)

result["messages"][-1].pretty_print()

---

### 🧪 Test 2: Ticket Creation with Approval ✅

This is the **main HITL demonstration**. It happens in 3 steps:

| Step | What happens |
|------|-------------|
| **A** | User asks to create a ticket → agent calls `create_ticket` → graph **pauses** |
| **B** | We inspect the interrupt to see what the agent wants to create |
| **C** | We approve with `Command(resume="yes")` → graph **resumes** → ticket created |

#### Step A: Send the request (graph will pause)

In [ ]:
# ============================================================
#  Test 2 — Step A: Send ticket request (graph pauses)
# ============================================================

ticket_config = {"configurable": {"thread_id": "test-ticket"}}

for step in helpdesk_graph.stream(
    {"messages": [{"role": "user", "content": "Create a support ticket. My monitor keeps flickering and nothing fixes it."}]},
    ticket_config
):
    for node_name, output in step.items():
        print(f"\n--- [{node_name}] ---")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n" + "=" * 50)
print("⏸️  Graph PAUSED — waiting for human approval...")
print("=" * 50)

#### Step B: Inspect the Interrupt

Before approving, let's look at what the graph is asking us.  
We use `get_state()` to peek into the frozen graph state.

In [ ]:
# ============================================================
#  Test 2 — Step B: Inspect the interrupt
# ============================================================

state = helpdesk_graph.get_state(ticket_config)

print("📋 Graph State Inspection")
print(f"   Next node(s): {state.next}")
print()

for task in state.tasks:
    if task.interrupts:
        for intr in task.interrupts:
            print("🛑 Interrupt Details:")
            print(f"   Question: {intr.value['question']}")
            print(f"   Summary:  {intr.value['summary']}")
            print(f"   Priority: {intr.value['priority']}")

#### Step C: Approve the Ticket ✅

We resume the graph with `Command(resume="yes")`.  
This unpauses the `create_ticket` tool — the `interrupt()` call returns `"yes"`, and the ticket gets created.

In [ ]:
# ============================================================
#  Test 2 — Step C: Approve (resume with "yes")
# ============================================================

for step in helpdesk_graph.stream(
    Command(resume="yes"),
    ticket_config
):
    for node_name, output in step.items():
        print(f"\n--- [{node_name}] ---")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n" + "=" * 50)
print("✅ Ticket created successfully!")
print("=" * 50)

---

### 🧪 Test 3: Ticket Creation with Rejection ❌

Same flow as Test 2, but this time we **reject** the ticket by resuming with `"no"`.  
The `create_ticket` tool will return a cancellation message instead.

#### Step A: Send the request (graph pauses)

In [ ]:
# ============================================================
#  Test 3 — Step A: Send ticket request
# ============================================================

reject_config = {"configurable": {"thread_id": "test-reject"}}

for step in helpdesk_graph.stream(
    {"messages": [{"role": "user", "content": "Please create a ticket. My keyboard is typing double letters."}]},
    reject_config
):
    for node_name, output in step.items():
        print(f"\n--- [{node_name}] ---")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n⏸️  Graph PAUSED — waiting for approval...")

#### Step B: Inspect the Interrupt

In [ ]:
# ============================================================
#  Test 3 — Step B: Inspect
# ============================================================

state = helpdesk_graph.get_state(reject_config)

print(f"Next node(s): {state.next}")
for task in state.tasks:
    if task.interrupts:
        for intr in task.interrupts:
            print(f"🛑 Interrupt: {intr.value}")

#### Step C: Reject the Ticket ❌

In [ ]:
# ============================================================
#  Test 3 — Step C: Reject (resume with "no")
# ============================================================

for step in helpdesk_graph.stream(
    Command(resume="no"),
    reject_config
):
    for node_name, output in step.items():
        print(f"\n--- [{node_name}] ---")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n" + "=" * 50)
print("❌ Ticket creation was rejected!")
print("=" * 50)

---

### 🧪 Test 4: View Tickets

Let's check what tickets exist in the store.  
We should see the ticket from Test 2 (approved) but **not** from Test 3 (rejected).

In [ ]:
# ============================================================
#  Test 4: View Tickets
# ============================================================

view_config = {"configurable": {"thread_id": "test-view"}}

result = helpdesk_graph.invoke(
    {"messages": [{"role": "user", "content": "Show me all open tickets"}]},
    view_config
)

result["messages"][-1].pretty_print()

---

### 🧪 Test 5: Multi-Turn Conversation Memory 🧠

This test demonstrates that the agent **remembers** previous messages within the same `thread_id`.  
We'll have a 4-turn conversation on a single thread:

| Turn | User says | What we're testing |
|------|-----------|-------------------|
| 1 | "My VPN keeps disconnecting" | Normal tool use |
| 2 | "That didn't work, create a ticket for this" | Agent remembers the VPN issue |
| 3 | Approve ticket | HITL resume |
| 4 | "Show me my tickets" | Ticket viewing |

In [ ]:
# ============================================================
#  Test 5 — Turn 1: Report a VPN issue
# ============================================================

memory_config = {"configurable": {"thread_id": "test-memory"}}

print("👤 User: My VPN keeps disconnecting every 10 minutes")
print("=" * 50)

result = helpdesk_graph.invoke(
    {"messages": [{"role": "user", "content": "My VPN keeps disconnecting every 10 minutes"}]},
    memory_config
)

result["messages"][-1].pretty_print()

In [ ]:
# ============================================================
#  Test 5 — Turn 2: Follow up → create a ticket
#  (Agent should remember the VPN issue from Turn 1!)
# ============================================================

print("👤 User: That didn't work, please create a ticket for this")
print("=" * 50)

for step in helpdesk_graph.stream(
    {"messages": [{"role": "user", "content": "That didn't work, please create a ticket for this"}]},
    memory_config
):
    for node_name, output in step.items():
        print(f"\n--- [{node_name}] ---")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n⏸️  Graph PAUSED — The agent remembered the VPN issue!")

In [ ]:
# ============================================================
#  Test 5 — Turn 2 (continued): Inspect & verify the summary
# ============================================================

state = helpdesk_graph.get_state(memory_config)

print(f"Next node(s): {state.next}")
for task in state.tasks:
    if task.interrupts:
        for intr in task.interrupts:
            print(f"\n🛑 Interrupt Details:")
            print(f"   Summary:  {intr.value['summary']}")
            print(f"   Priority: {intr.value['priority']}")
            print(f"\n   👆 Notice: The agent used context from Turn 1 to fill the summary!")

In [ ]:
# ============================================================
#  Test 5 — Turn 3: Approve the ticket
# ============================================================

print("👤 Human: Approving ticket...")
print("=" * 50)

for step in helpdesk_graph.stream(
    Command(resume="yes"),
    memory_config
):
    for node_name, output in step.items():
        print(f"\n--- [{node_name}] ---")
        for msg in output.get("messages", []):
            msg.pretty_print()

print("\n✅ Ticket created via multi-turn conversation!")

In [ ]:
# ============================================================
#  Test 5 — Turn 4: View tickets (same thread)
# ============================================================

print("👤 User: Show me my tickets")
print("=" * 50)

result = helpdesk_graph.invoke(
    {"messages": [{"role": "user", "content": "Show me my tickets"}]},
    memory_config
)

result["messages"][-1].pretty_print()

---

## 🗄️ Step 13: Inspect the Ticket Store

Let's look at the raw ticket data to confirm everything was stored correctly.

In [ ]:
# ============================================================
#  Inspect the raw ticket store
# ============================================================

print("🗄️  All Tickets in Store")
print("=" * 60)

tickets = get_all_tickets()

if not tickets:
    print("   (No tickets found)")
else:
    for i, t in enumerate(tickets, 1):
        print(f"\n  Ticket #{i}:")
        print(f"    ID:       {t['ticket_id']}")
        print(f"    Summary:  {t['summary']}")
        print(f"    Priority: {t['priority']}")
        print(f"    Status:   {t['status']}")
        print(f"    Created:  {t['created_at']}")

print(f"\n  Total tickets: {len(tickets)}")

---

## 📊 Full Series Comparison: V1 → V2 → V3 → V4

| Feature | V1 | V2 | V3 | V4 |
|---------|:--:|:--:|:--:|:--:|
| **Routing** | Manual `if/else` | LLM classification | LLM tool calls | LLM tool calls |
| **State** | `TypedDict` | `TypedDict` | `MessagesState` | `MessagesState` |
| **LLM Usage** | Per-node responses | Classifier + responders | Single agent + tools | Single agent + tools |
| **Tools** | ❌ | ❌ | ✅ 3 tools | ✅ 5 tools |
| **Tool Loop** | ❌ | ❌ | ✅ `tools_condition` | ✅ `tools_condition` |
| **Ticket Creation** | ❌ | ❌ | ❌ | ✅ `create_ticket` |
| **Ticket Viewing** | ❌ | ❌ | ❌ | ✅ `view_tickets` |
| **Human-in-the-Loop** | ❌ | ❌ | ❌ | ✅ `interrupt()` |
| **Memory** | ❌ | ❌ | ❌ | ✅ `MemorySaver` |
| **Multi-turn** | ❌ | ❌ | ❌ | ✅ `thread_id` |
| **Checkpointer** | ❌ | ❌ | ❌ | ✅ Required for HITL |

### 🎓 Concepts Learned at Each Stage

| Version | Key Concepts |
|---------|-------------|
| **V1** | `StateGraph`, `TypedDict`, `add_node`, `add_edge`, `add_conditional_edges`, `START`, `END` |
| **V2** | LLM-powered routing, structured output, prompt engineering for classification |
| **V3** | `@tool` decorator, `ToolNode`, `tools_condition`, `bind_tools`, `MessagesState`, agent loop pattern |
| **V4** | `MemorySaver` checkpointer, `interrupt()`, `Command(resume=...)`, `get_state()`, `thread_id`, HITL pattern |

---

## ✅ Summary

In this notebook, we extended our V3 agent with three powerful new capabilities:

### 🎫 Ticket Creation
- Added `create_ticket` and `view_tickets` tools
- Built an in-memory `TICKET_STORE` for persistence

### 🛑 Human-in-the-Loop
- Used `interrupt()` inside `create_ticket` to **pause** the graph
- Used `get_state()` to **inspect** what the graph is waiting for
- Used `Command(resume=...)` to **approve or reject** the action

### 🧠 Conversation Memory
- Added `MemorySaver` as the graph's checkpointer
- Used `thread_id` to maintain conversation history across turns
- Demonstrated the agent referencing earlier messages in later turns

### 🔑 Key Takeaway

> The **graph structure didn't change** from V3 — it's still `agent ↔ tools` in a loop.  
> The new capabilities come from the **checkpointer** (enables memory + HITL) and  
> the **`interrupt()` call inside the tool** (enables human approval).  
> This is the power of LangGraph's composable design!

---

*AI IT Helpdesk Agent — Version 4: Human-in-the-Loop + Tickets + Memory*